# 🦙 Ollama over NetBird Quickstart Guide

This notebook demonstrates how to connect to the **AIT Brainlab Ollama Service** on server `la` across the **NetBird WireGuard Mesh** (`http://la:11434`).

### Prerequisites:
- Connected to NetBird VPN (`https://netbird.brain.cs.ait.ac.th`)
- `pip install ollama pandas`

> **Note for JupyterLab on Server `la`**: If running inside a container on server `la` itself, you can also use `http://ollama:11434`.

In [22]:
# Install prerequisites
%pip install -q ollama pandas

/mnt/pool-1/home/akraradets/projects/AIT-brainlab/brainlab-base/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [23]:
from ollama import Client
import pandas as pd

# Initialize client targeting server 'la' over NetBird MagicDNS
# (If inside a JupyterHub container on 'la', use 'http://ollama:11434')
OLLAMA_HOST = "http://la:11434"
client = Client(host=OLLAMA_HOST)

print(f"Connected to Ollama at {OLLAMA_HOST}")

Connected to Ollama at http://la:11434


## 1. List Available Models
View all models currently downloaded and available on the server (`client.list()`).

In [24]:
models_resp = client.list()

model_rows = []
for m in models_resp.get("models", []):
    size = (m.get("size") if isinstance(m, dict) else getattr(m, "size", 0)) or 0
    size_gb = size / (1024 ** 3)
    details = (m.get("details") if isinstance(m, dict) else getattr(m, "details", {})) or {}
    modified = m.get("modified_at") if isinstance(m, dict) else getattr(m, "modified_at", None)
    
    model_name = (m.get("model") or m.get("name")) if isinstance(m, dict) else (getattr(m, "model", None) or getattr(m, "name", "N/A"))
    family = details.get("family") if isinstance(details, dict) else getattr(details, "family", "N/A")
    param_size = details.get("parameter_size") if isinstance(details, dict) else getattr(details, "parameter_size", "N/A")
    quant = details.get("quantization_level") if isinstance(details, dict) else getattr(details, "quantization_level", "N/A")
    
    model_rows.append({
        "Model": model_name,
        "Size": f"{size_gb:.2f} GB",
        "Family": family or "N/A",
        "Parameter Size": param_size or "N/A",
        "Quantization": quant or "N/A",
        "Modified": str(modified)[:19] if modified else "N/A",
    })

df_models = pd.DataFrame(model_rows)
display(df_models)

,Model,Size,Family,Parameter Size,Quantization,Modified
0,qwen3.8:27b,16.52 GB,qwen35,27.3B,Q4_K_M,2026-11-10 18:36:19
1,llama3.2:1b,1.23 GB,llama,1.2B,Q8_0,2026-11-10 18:27:28


## 2. See Active (Loaded) Models in GPU VRAM
Check which models are currently consuming GPU memory (`client.ps()`).

In [25]:
ps_resp = client.ps()

active_rows = []
for m in ps_resp.get("models", []):
    size_vram_gb = m.get("size_vram", 0) / (1024 ** 3)
    active_rows.append({
        "Model": m.get("name") or m.get("model"),
        "VRAM Used": f"{size_vram_gb:.2f} GB",
        "Expires At": m.get("expires_at", "N/A"),
    })

if active_rows:
    display(pd.DataFrame(active_rows))
else:
    print("ℹ️ No models are currently loaded in GPU VRAM (Ollama is idle).")

,Model,VRAM Used,Expires At
0,qwen3.8:27b,17.00 GB,2026-11-10 19:37:48.094744+00:00


## 3. Download / Pull a Model
Download a model directly from the Ollama library (`client.pull()`).

*Note: The server pulls models through the institutional CSIM forward proxy (`http://192.41.170.82:3128`).*

In [26]:
# Target model to download
# target_model = "llama3.2:1b"
target_model = "qwen3.8:27b"
print(f"Pulling '{target_model}'...\n")

for chunk in client.pull(target_model, stream=True):
    status = chunk.get("status", "") if isinstance(chunk, dict) else getattr(chunk, "status", "")
    total = (chunk.get("total") if isinstance(chunk, dict) else getattr(chunk, "total", 0)) or 0
    completed = (chunk.get("completed") if isinstance(chunk, dict) else getattr(chunk, "completed", 0)) or 0
    
    if total > 0:
        pct = (completed / total) * 100
        print(f"\r{status}: {pct:5.1f}% ({completed / 1024**2:.1f} MB / {total / 1024**2:.1f} MB)", end="", flush=True)
    else:
        print(f"\r{status}...", end="", flush=True)

print("\n\n✅ Model ready!")

Pulling 'qwen3.8:27b'...

success...nused layers....0% (0.0 MB / 0.0 MB)31.9 MB)

✅ Model ready!


## 4. Select / Switch Models (and Explicitly Unload)
Our Ollama instance is configured with `OLLAMA_MAX_LOADED_MODELS=1` to enforce our 50% GPU slice (24 GB ceiling).
- Simply calling a new model automatically evicts the previous one from VRAM.
- You can also **explicitly unload** a model immediately by passing `keep_alive=0`.

In [27]:
def unload_model(model_name: str):
    """Instantly frees GPU VRAM by evicting the model from memory."""
    client.generate(model=model_name, prompt="", keep_alive=0)
    print(f"🗑️ Evicted '{model_name}' from GPU VRAM.")

# Check active models and unload if necessary
current_active = client.ps().get("models", [])
for m in current_active:
    name = m.get("name") or m.get("model")
    unload_model(name)

# Select the active model for our tests
# SELECTED_MODEL = "llama3.2:1b"
SELECTED_MODEL = "qwen3.8:27b"
print(f"🎯 Selected Model: {SELECTED_MODEL}")

🗑️ Evicted 'qwen3.8:27b' from GPU VRAM.
🎯 Selected Model: qwen3.8:27b


## 5. Use the Model (Chat Completion with Streaming)
Send prompt messages to the model and stream token output in real-time.

In [28]:
messages = [
    {"role": "system", "content": "You are a concise AI research assistant in AIT Brainlab."},
    {"role": "user", "content": "Explain the difference between full fine-tuning and LoRA in 3 bullet points."}
]

print(f"🤖 Streaming response from {SELECTED_MODEL}:\n")
stream = client.chat(model=SELECTED_MODEL, messages=messages, stream=True)

for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print("\n")

🤖 Streaming response from qwen3.8:27b:

- **Parameters updated:** Full fine-tuning updates *every* weight in the model (billions to trillions), while LoRA freezes the base model and only learns small low-rank adapter matrices (A, B) injected into a few layers (e.g., attention projections)—typically 0.1–1 % of the total parameters.

- **Memory & compute cost:** Full fine-tuning requires storing optimizer states (Adam moments, gradients) for all weights, making it memory-intensive (often 10–16× the model size in training memory). LoRA's trainable set is tiny, so it can run on a single GPU or even a laptop, with training time dropping by an order of magnitude.

- **Swappability & deployment:** With LoRA, the base checkpoint stays intact and you can hot-swap or stack different adapters per task without re-downloading the model; full fine-tuning bakes the adaptation into a single monolithic checkpoint, so each task needs its own full model copy.



## 6. Change Parameters & Keep-Alive (Make Model Active Longer)
You can customize inference parameters and control memory residency per-request:
- **`keep_alive`**: Extends how long the model stays loaded in GPU VRAM (e.g. `"1h"`, `"30m"`, or `-1` for indefinitely).
- **`options`**: Set `temperature`, `num_ctx` (context window size), `top_p`, etc.

In [29]:
# Call the model with custom parameters and a 1-hour VRAM keep-alive
response = client.chat(
    model=SELECTED_MODEL,
    messages=[
        {"role": "user", "content": "Write a haiku about GPU clusters in an academic lab."}
    ],
    options={
        "temperature": 0.3,    # Lower for more focused, deterministic output
        "top_p": 0.9,
        "num_ctx": 4096,       # Allocate 4096 tokens context window
    },
    keep_alive="1h"            # Keep model loaded in GPU VRAM for 1 hour!
)

print("Response:")
print(response["message"]["content"])

# Verify that the expiration time in VRAM is now extended
print("\n--- Active VRAM Status ---")
for m in client.ps().get("models", []):
    vram_gb = m.get("size_vram", 0) / (1024 ** 3)
    print(f"Model: {m.get('name')} | VRAM: {vram_gb:.2f} GB | Expires At: {m.get('expires_at')}")

Response:
Rows of green LEDs
humming, blinking, learning deep—
the grad student sleeps

--- Active VRAM Status ---
Model: qwen3.8:27b | VRAM: 17.00 GB | Expires At: 2026-11-10 19:39:49.148676+00:00
